In [ ]:
import MODE_utilities as AS

import os
import gc
import copy
import random
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)

import optuna

In [ ]:
# ============================================================
# CLIENT-COUNT SCALABILITY SETTINGS
# Raspberry Pi CPU, fixed T = 500
# ============================================================

SERIES_LENGTH = 500

N_VALUES = [
    10,
    100,
    500,
    1000,
]

TYPE_OF_DATA = "cpu"


# ============================================================
# External offline / online split
# ============================================================

# First 80% of every profile = offline development/HPO data
OFFLINE_FRACTION = 0.80

# Within the offline part:
# first 70% = training / LP-Mirror source candidates
# last 30%  = validation / LP-Mirror guidance set
HPO_SPLIT = 0.70

NORMALIZE = True


# ============================================================
# LSTM hyperparameter search space
# Same as original MOTPE+LP-Mirror benchmark
# ============================================================

WINDOW_MIN = 5
WINDOW_MAX = 20

LR_MIN = 0.0001
LR_MAX = 0.1

BATCH_MIN = 8
BATCH_MAX = 128

HIDDEN_MIN = 2
HIDDEN_MAX = 10

EPOCH_MIN = 50
EPOCH_MAX = 150

LAYER_MIN = 1
LAYER_MAX = 3


# ============================================================
# MOTPE / Optuna
# ============================================================

N_TRIALS = 100

N_STARTUP_TRIALS = 10

N_EI_CANDIDATES = 24

SEED = 89


# ============================================================
# LP-Mirror settings
# Same as original benchmark
# ============================================================

LP_SEQ_LEN = WINDOW_MAX

LP_PRED_LEN = 1

LP_BATCH_SIZE = 128

LP_PROXY_LR = 0.01
LP_PROXY_EPOCHS = 1

LP_ADAPT_LR = 0.01
LP_ADAPT_STEPS = 1

LP_POSITIVE_THRESHOLD = 1e-6


# ============================================================
# Final Pareto preference
# ============================================================

FINAL_WEIGHTS = np.array(
    [0.6, 0.4],
    dtype=float,
)


# ============================================================
# Output folder
# ============================================================

OUTPUT_ROOT = (
    "MOTPE_LP_Mirror_scalability_results"
)

OUTPUT_FOLDER = os.path.join(
    OUTPUT_ROOT,
    "cpu",
    "Part_A_Number_of_Series",
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True,
)

In [ ]:
def set_all_seeds(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_all_seeds()

print("Seed:", SEED)
print("Device:", AS.device)
print("Optuna version:", optuna.__version__)

In [ ]:
# ============================================================
# Load ALL available Raspberry Pi CPU series of length 500
# Exactly as in MODE client-count scalability
# ============================================================

all_raspberry_cpu = (
    AS.read_raspberry_pi_dataset(
        series_length=SERIES_LENGTH,
        column="cpu",
    )
)

print(
    "Available complete CPU time series:",
    len(all_raspberry_cpu),
)

print(
    "Maximum requested N:",
    max(N_VALUES),
)

if len(all_raspberry_cpu) < max(N_VALUES):

    raise ValueError(
        f"Need at least {max(N_VALUES)} complete "
        f"time series of length {SERIES_LENGTH}, "
        f"but only {len(all_raspberry_cpu)} are available."
    )

In [ ]:
def prepare_experiment_data(
    data,
    series_length=SERIES_LENGTH,
    offline_fraction=OFFLINE_FRACTION,
):

    data = np.asarray(
        data,
        dtype=np.float64,
    )

    # ============================================================
    # Flat data
    # ============================================================

    if data.ndim == 1:

        num_full_series = (
            len(data) // series_length
        )

        trimmed_data = data[
            :num_full_series * series_length
        ]

        split_data = trimmed_data.reshape(
            num_full_series,
            series_length,
        )

    # ============================================================
    # Already separated profiles: Raspberry Pi
    # ============================================================

    elif data.ndim == 2:

        if data.shape[1] != series_length:

            raise ValueError(
                f"Expected series length {series_length}, "
                f"but received shape {data.shape}."
            )

        split_data = data.copy()

    else:

        raise ValueError(
            f"Expected 1-D or 2-D data, "
            f"got shape {data.shape}."
        )

    # ============================================================
    # Offline / online
    # ============================================================

    offline_length = int(
        offline_fraction * series_length
    )

    offline_data = split_data[
        :, :offline_length
    ]

    online_data = split_data[
        :, offline_length:
    ]

    return (
        split_data,
        offline_data,
        online_data,
    )

In [ ]:
class LPMirrorLinearProxy(nn.Module):

    def __init__(
        self,
        seq_len,
        pred_len=1,
    ):
        super().__init__()

        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)

        self.linear = nn.Linear(
            self.seq_len,
            self.pred_len,
        )

    def forward(self, x):

        # x:
        # [batch, time, channels]

        x = x.permute(
            0,
            2,
            1,
        )

        x = self.linear(x)

        x = x.permute(
            0,
            2,
            1,
        )

        return x

In [ ]:
def build_proxy_samples(
    series_data,
    seq_len,
    segment_start,
    segment_end,
):
    """
    Create one-step-ahead samples entirely contained inside
    [segment_start, segment_end).

    Returns
    -------
    X : shape (n_samples, seq_len, 1)
    y : shape (n_samples, 1, 1)
    metadata : shape (n_samples, 2)
        columns = [series_id, target_time]
    """

    series_data = np.asarray(
        series_data,
        dtype=np.float64,
    )

    X = []
    y = []
    metadata = []

    n_series = series_data.shape[0]

    first_target = (
        segment_start + seq_len
    )

    for series_id in range(
        n_series
    ):

        ts = series_data[
            series_id
        ]

        for target_time in range(
            first_target,
            segment_end,
        ):

            x = ts[
                target_time - seq_len:
                target_time
            ]

            target = ts[
                target_time
            ]

            X.append(x)
            y.append(target)

            metadata.append(
                (
                    series_id,
                    target_time,
                )
            )

    X = np.asarray(
        X,
        dtype=np.float32,
    )[..., None]

    y = np.asarray(
        y,
        dtype=np.float32,
    )[:, None, None]

    metadata = np.asarray(
        metadata,
        dtype=np.int64,
    )

    return X, y, metadata

In [ ]:
def run_lp_mirror_selection(
    offline_scaled,
):

    set_all_seeds(SEED)

    offline_scaled = np.asarray(
        offline_scaled,
        dtype=np.float64,
    )

    n_series, offline_length = (
        offline_scaled.shape
    )

    train_end = int(
        HPO_SPLIT * offline_length
    )

    # ============================================================
    # Source candidates
    # ============================================================

    (
        X_source,
        y_source,
        source_metadata,
    ) = build_proxy_samples(
        series_data=offline_scaled,
        seq_len=LP_SEQ_LEN,
        segment_start=0,
        segment_end=train_end,
    )

    # ============================================================
    # Validation / guidance samples
    # ============================================================

    (
        X_guide,
        y_guide,
        guide_metadata,
    ) = build_proxy_samples(
        series_data=offline_scaled,
        seq_len=LP_SEQ_LEN,
        segment_start=train_end,
        segment_end=offline_length,
    )

    print(
        "LP-Mirror candidate samples:",
        len(X_source),
    )

    print(
        "LP-Mirror guidance samples:",
        len(X_guide),
    )

    source_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_source),
            torch.tensor(y_source),
        ),
        batch_size=LP_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
    )

    guide_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_guide),
            torch.tensor(y_guide),
        ),
        batch_size=LP_BATCH_SIZE,
        shuffle=True,
        drop_last=False,
    )

    # ============================================================
    # Start LP-Mirror timer
    # ============================================================

    AS._sync_device()

    selection_start = (
        time.perf_counter()
    )

    # ============================================================
    # 1. Proxy
    # ============================================================

    proxy = LPMirrorLinearProxy(
        seq_len=LP_SEQ_LEN,
        pred_len=LP_PRED_LEN,
    ).to(
        AS.device
    )

    optimizer = optim.Adam(
        proxy.parameters(),
        lr=LP_PROXY_LR,
    )

    criterion = nn.MSELoss()

    proxy.train()

    for _ in range(
        LP_PROXY_EPOCHS
    ):

        for X_batch, y_batch in (
            source_loader
        ):

            X_batch = X_batch.to(
                AS.device
            )

            y_batch = y_batch.to(
                AS.device
            )

            optimizer.zero_grad()

            pred = proxy(
                X_batch
            )

            loss = criterion(
                pred,
                y_batch,
            )

            loss.backward()

            optimizer.step()

    # ============================================================
    # 2. Save original proxy
    # ============================================================

    original_proxy = copy.deepcopy(
        proxy
    )

    # ============================================================
    # 3. Adapt proxy on validation/guidance data
    # ============================================================

    adapt_optimizer = optim.SGD(
        proxy.parameters(),
        lr=LP_ADAPT_LR,
    )

    proxy.train()

    for _ in range(
        LP_ADAPT_STEPS
    ):

        for X_batch, y_batch in (
            guide_loader
        ):

            X_batch = X_batch.to(
                AS.device
            )

            y_batch = y_batch.to(
                AS.device
            )

            adapt_optimizer.zero_grad()

            pred = proxy(
                X_batch
            )

            loss = criterion(
                pred,
                y_batch,
            )

            loss.backward()

            adapt_optimizer.step()

    adapted_proxy = proxy

    # ============================================================
    # 4. Influence scores
    # ============================================================

    original_proxy.eval()
    adapted_proxy.eval()

    all_scores = []

    with torch.inference_mode():

        for X_batch, y_batch in (
            source_loader
        ):

            X_batch = X_batch.to(
                AS.device
            )

            y_batch = y_batch.to(
                AS.device
            )

            pred_original = (
                original_proxy(
                    X_batch
                )
            )

            pred_adapted = (
                adapted_proxy(
                    X_batch
                )
            )

            loss_original = (
                (
                    pred_original
                    - y_batch
                ) ** 2
            ).mean(
                dim=(1, 2)
            )

            loss_adapted = (
                (
                    pred_adapted
                    - y_batch
                ) ** 2
            ).mean(
                dim=(1, 2)
            )

            # LP-Mirror Forward-INF score:
            # positive = beneficial
            scores = (
                loss_original
                - loss_adapted
            )

            all_scores.append(
                scores.cpu().numpy()
            )

    all_scores = np.concatenate(
        all_scores
    )

    # ============================================================
    # 5. Positive selection
    # ============================================================

    selected_mask = (
        all_scores
        > LP_POSITIVE_THRESHOLD
    )

    selected_indices = np.where(
        selected_mask
    )[0]

    selected_targets = (
        source_metadata[
            selected_indices
        ]
    )

    if len(selected_targets) == 0:

        raise RuntimeError(
            "LP-Mirror selected zero samples. "
            "The experiment is stopped rather than "
            "silently changing the selection method."
        )

    AS._sync_device()

    selection_duration = (
        time.perf_counter()
        - selection_start
    )

    original_size = len(
        source_metadata
    )

    selected_size = len(
        selected_targets
    )

    retention_ratio = (
        selected_size
        / original_size
    )

    print(
        f"LP-Mirror selected "
        f"{selected_size}/{original_size} "
        f"samples "
        f"({100 * retention_ratio:.2f}%)."
    )

    print(
        "LP-Mirror selection time [s]:",
        selection_duration,
    )

    # Free proxy models
    del proxy
    del original_proxy
    del adapted_proxy

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "selected_targets":
            selected_targets,

        "selected_indices":
            selected_indices,

        "scores":
            all_scores,

        "original_train_size":
            int(original_size),

        "selected_train_size":
            int(selected_size),

        "retention_ratio":
            float(retention_ratio),

        "selection_duration":
            float(selection_duration),

        "train_end":
            int(train_end),

        "offline_length":
            int(offline_length),

        "guide_size":
            int(len(guide_metadata)),
    }

In [ ]:
def build_selected_lstm_samples(
    offline_scaled,
    selected_targets,
    window,
):

    X = []
    y = []

    for series_id, target_time in (
        selected_targets
    ):

        series_id = int(
            series_id
        )

        target_time = int(
            target_time
        )

        x = offline_scaled[
            series_id,
            target_time - window:
            target_time,
        ]

        if len(x) != window:
            continue

        X.append(x)

        y.append(
            offline_scaled[
                series_id,
                target_time,
            ]
        )

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    y = np.asarray(
        y,
        dtype=np.float32,
    )

    return X, y

In [ ]:
def build_lstm_validation_samples(
    offline_scaled,
    window,
    validation_start,
):

    X = []
    y = []

    n_series = (
        offline_scaled.shape[0]
    )

    end_time = (
        offline_scaled.shape[1]
    )

    # Require each validation window to be
    # completely inside validation.
    first_target = (
        validation_start
        + window
    )

    for series_id in range(
        n_series
    ):

        for target_time in range(
            first_target,
            end_time,
        ):

            X.append(
                offline_scaled[
                    series_id,
                    target_time - window:
                    target_time,
                ]
            )

            y.append(
                offline_scaled[
                    series_id,
                    target_time,
                ]
            )

    return (
        np.asarray(
            X,
            dtype=np.float32,
        ),
        np.asarray(
            y,
            dtype=np.float32,
        ),
    )

In [ ]:
class LPMirrorLSTMInstance:

    def __init__(
        self,
        model,
        chromosome,
    ):

        self.model = model

        self.chromosome = (
            chromosome
        )

        self.NORMAL = True

In [ ]:
def train_evaluate_lp_lstm(
    offline_scaled,
    selected_targets,
    architecture,
    validation_start,
):

    window = int(
        architecture["window"]
    )

    batch_size = int(
        architecture["batch_size"]
    )

    learning_rate = float(
        architecture["learning_rate"]
    )

    hidden_dim = int(
        architecture["hidden_dim"]
    )

    epochs = int(
        architecture["epochs"]
    )

    layer_dim = int(
        architecture["layer_dim"]
    )

    # ============================================================
    # Selected LP-Mirror training samples
    # ============================================================

    X_train, y_train = (
        build_selected_lstm_samples(
            offline_scaled=offline_scaled,
            selected_targets=selected_targets,
            window=window,
        )
    )

    # ============================================================
    # Complete validation set
    # ============================================================

    X_val, y_val = (
        build_lstm_validation_samples(
            offline_scaled=offline_scaled,
            window=window,
            validation_start=validation_start,
        )
    )

    if len(X_train) == 0:

        raise ValueError(
            "No selected training samples "
            "for this architecture."
        )

    if len(X_val) == 0:

        raise ValueError(
            "No validation samples "
            "for this architecture."
        )

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train),
            torch.tensor(y_train),
        ),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_val),
            torch.tensor(y_val),
        ),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    # ============================================================
    # Same LSTM used by MODE
    # ============================================================

    model = AS.LSTMModel(
        input_dim=1,
        hidden_dim=hidden_dim,
        layer_dim=layer_dim,
        output_dim=1,
    ).to(
        AS.device
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    loss_fn = nn.MSELoss()

    # ============================================================
    # TRAINING TIME objective
    # ============================================================

    AS._sync_device()

    train_start = (
        time.perf_counter()
    )

    model.train()

    for _ in range(
        epochs
    ):

        for X_batch, y_batch in (
            train_loader
        ):

            X_batch = (
                X_batch
                .unsqueeze(-1)
                .to(AS.device)
            )

            y_batch = (
                y_batch
                .unsqueeze(1)
                .to(AS.device)
            )

            optimizer.zero_grad()

            pred = model(
                X_batch
            )

            loss = loss_fn(
                pred,
                y_batch,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

    AS._sync_device()

    training_time = (
        time.perf_counter()
        - train_start
    )

    # ============================================================
    # Validation RMSE
    # ============================================================

    preds = []
    targets = []

    model.eval()

    with torch.inference_mode():

        for X_batch, y_batch in (
            val_loader
        ):

            X_batch = (
                X_batch
                .unsqueeze(-1)
                .to(AS.device)
            )

            output = model(
                X_batch
            )

            preds.append(
                output
                .cpu()
                .numpy()
                .reshape(-1)
            )

            targets.append(
                y_batch
                .numpy()
                .reshape(-1)
            )

    preds = np.concatenate(
        preds
    )

    targets = np.concatenate(
        targets
    )

    val_rmse = float(
        np.sqrt(
            np.mean(
                (
                    preds
                    - targets
                ) ** 2
            )
        )
    )

    instance = (
        LPMirrorLSTMInstance(
            model=model,
            chromosome=architecture,
        )
    )

    return (
        val_rmse,
        float(training_time),
        instance,
        len(X_train),
    )

In [ ]:
def run_motpe_hpo(
    offline_scaled,
    selected_targets,
    validation_start,
):

    set_all_seeds(
        SEED
    )

    trial_models = {}

    # ============================================================
    # Modern Optuna MOTPE
    # ============================================================

    sampler = (
        optuna.samplers.TPESampler(
            n_startup_trials=N_STARTUP_TRIALS,
            n_ei_candidates=N_EI_CANDIDATES,
            seed=SEED,
        )
    )

    study = optuna.create_study(
        directions=[
            "minimize",
            "minimize",
        ],
        sampler=sampler,
    )

    # ============================================================
    # Objective
    # ============================================================

    def objective(trial):

        architecture = {
            "window":
                trial.suggest_int(
                    "window",
                    WINDOW_MIN,
                    WINDOW_MAX,
                ),

            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    LR_MIN,
                    LR_MAX,
                ),

            "batch_size":
                trial.suggest_int(
                    "batch_size",
                    BATCH_MIN,
                    BATCH_MAX,
                ),

            "hidden_dim":
                trial.suggest_int(
                    "hidden_dim",
                    HIDDEN_MIN,
                    HIDDEN_MAX,
                ),

            "epochs":
                trial.suggest_int(
                    "epochs",
                    EPOCH_MIN,
                    EPOCH_MAX,
                ),

            "layer_dim":
                trial.suggest_int(
                    "layer_dim",
                    LAYER_MIN,
                    LAYER_MAX,
                ),
        }

        (
            val_rmse,
            training_time,
            instance,
            selected_size,
        ) = train_evaluate_lp_lstm(
            offline_scaled=offline_scaled,
            selected_targets=selected_targets,
            architecture=architecture,
            validation_start=validation_start,
        )

        # Store already-trained candidate
        instance.model = (
            instance.model.to("cpu")
        )

        trial_models[
            trial.number
        ] = instance

        trial.set_user_attr(
            "selected_train_size",
            int(selected_size),
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return (
            val_rmse,
            training_time,
        )

    # ============================================================
    # HPO timer
    # ============================================================

    AS._sync_device()

    start = time.perf_counter()

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        n_jobs=1,
        gc_after_trial=True,
        show_progress_bar=True,
    )

    AS._sync_device()

    hpo_duration = (
        time.perf_counter()
        - start
    )

    return (
        study,
        trial_models,
        float(hpo_duration),
    )

In [ ]:
def select_final_motpe_trial(
    study,
):

    pareto_trials = (
        study.best_trials
    )

    values = np.asarray(
        [
            trial.values
            for trial in pareto_trials
        ],
        dtype=float,
    )

    rmse = values[:, 0]
    train_time = values[:, 1]

    rmse_n = (
        (rmse - rmse.min())
        /
        (
            rmse.max()
            - rmse.min()
            + 1e-12
        )
    )

    time_n = (
        (train_time - train_time.min())
        /
        (
            train_time.max()
            - train_time.min()
            + 1e-12
        )
    )

    score = (
        FINAL_WEIGHTS[0]
        * rmse_n
        +
        FINAL_WEIGHTS[1]
        * time_n
    )

    best_position = int(
        np.argmin(score)
    )

    best_trial = (
        pareto_trials[
            best_position
        ]
    )

    return (
        best_trial,
        pareto_trials,
    )

In [ ]:
def run_lp_mirror_motpe_experiment(
    experiment_name,
    data,
    output_folder=OUTPUT_FOLDER,
):

    os.makedirs(
        output_folder,
        exist_ok=True,
    )

    set_all_seeds(
        SEED
    )

    print("\n" + "=" * 80)
    print(
        f"STARTING: {experiment_name}"
    )
    print("=" * 80)

    # ============================================================
    # TOTAL PIPELINE TIMER
    # ============================================================

    AS._sync_device()

    total_start = (
        time.perf_counter()
    )

    # ============================================================
    # 1. Prepare data
    # ============================================================

    (
        split_data,
        offline_data,
        online_data,
    ) = prepare_experiment_data(
        data=data,
    )

    # Per-profile normalization exactly as original benchmark
    offline_scaled = (
        AS.minmax_normalize_ts(
            offline_data
        )
    )

    print(
        "Complete shape:",
        split_data.shape,
    )

    print(
        "Offline shape:",
        offline_data.shape,
    )

    print(
        "Online shape:",
        online_data.shape,
    )

    # ============================================================
    # 2. LP-Mirror selection
    # ============================================================

    lp_result = (
        run_lp_mirror_selection(
            offline_scaled
        )
    )

    selected_targets = (
        lp_result[
            "selected_targets"
        ]
    )

    selected_train_size = (
        lp_result[
            "selected_train_size"
        ]
    )

    original_train_size = (
        lp_result[
            "original_train_size"
        ]
    )

    retention_ratio = (
        lp_result[
            "retention_ratio"
        ]
    )

    lp_selection_duration = (
        lp_result[
            "selection_duration"
        ]
    )

    validation_start = (
        lp_result[
            "train_end"
        ]
    )

    # ============================================================
    # 3. MOTPE HPO
    # ============================================================

    (
        study,
        trial_models,
        motpe_duration,
    ) = run_motpe_hpo(
        offline_scaled=offline_scaled,
        selected_targets=selected_targets,
        validation_start=validation_start,
    )

    # Optimization duration =
    # LP-Mirror selection + MOTPE HPO + minor pipeline overhead
    AS._sync_device()

    total_optimization_duration = (
        time.perf_counter()
        - total_start
    )

    # ============================================================
    # 4. Select final Pareto trial
    # ============================================================

    (
        best_trial,
        pareto_trials,
    ) = select_final_motpe_trial(
        study
    )

    best_instance = (
        trial_models[
            best_trial.number
        ]
    )

    best_instance.model = (
        best_instance.model.to(
            AS.device
        )
    )

    best_architecture = {
        "window":
            int(
                best_trial.params[
                    "window"
                ]
            ),

        "learning_rate":
            float(
                best_trial.params[
                    "learning_rate"
                ]
            ),

        "batch_size":
            int(
                best_trial.params[
                    "batch_size"
                ]
            ),

        "hidden_dim":
            int(
                best_trial.params[
                    "hidden_dim"
                ]
            ),

        "epochs":
            int(
                best_trial.params[
                    "epochs"
                ]
            ),

        "layer_dim":
            int(
                best_trial.params[
                    "layer_dim"
                ]
            ),
    }

    # ============================================================
    # 5. Held-out ONLINE evaluation
    # ============================================================

    online_results = []

    for series_id, ts in enumerate(
        online_data
    ):

        result = (
            AS.evaluate_one_individual_series(
                ML=best_instance,
                series_raw=ts,
                sdtw_normalize=NORMALIZE,
                return_raw_scale=True,
            )
        )

        online_results.append({
            "series_id":
                int(series_id),

            "rmse_normalized":
                float(
                    result[
                        "rmse_used"
                    ]
                ),

            "rmse_raw":
                float(
                    result[
                        "rmse_raw"
                    ]
                ),

            "nrmse_raw":
                float(
                    result[
                        "nrmse_percent_raw"
                    ]
                ),
        })

    raw_rmse_values = np.asarray(
        [
            row["rmse_raw"]
            for row in online_results
        ],
        dtype=float,
    )

    normalized_rmse_values = np.asarray(
        [
            row["rmse_normalized"]
            for row in online_results
        ],
        dtype=float,
    )

    # ============================================================
    # END-TO-END duration including online evaluation
    # ============================================================

    AS._sync_device()

    total_end_to_end_duration = (
        time.perf_counter()
        - total_start
    )

    # ============================================================
    # 6. Additional data-size quantity
    # ============================================================

    selected_input_values = (
        selected_train_size
        * best_architecture[
            "window"
        ]
    )

    # ============================================================
    # 7. Save TXT
    # ============================================================

    txt_path = os.path.join(
        output_folder,
        (
            f"LP_Mirror_MOTPE_"
            f"{experiment_name}_results.txt"
        ),
    )

    with open(
        txt_path,
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            f"LP-MIRROR + MOTPE - "
            f"{experiment_name}\n"
        )

        f.write("=" * 72 + "\n")

        # --------------------------------------------------------
        # Settings
        # --------------------------------------------------------

        f.write(
            "\nEXPERIMENT SETTINGS\n"
        )

        f.write("-" * 72 + "\n")

        f.write(
            f"Seed: {SEED}\n"
        )

        f.write(
            f"MOTPE trials: "
            f"{N_TRIALS}\n"
        )

        f.write(
            f"MOTPE startup trials: "
            f"{N_STARTUP_TRIALS}\n"
        )

        f.write(
            f"LP sequence length: "
            f"{LP_SEQ_LEN}\n"
        )

        f.write(
            f"LP prediction length: "
            f"{LP_PRED_LEN}\n"
        )

        f.write(
            f"LP adaptation LR: "
            f"{LP_ADAPT_LR}\n"
        )

        f.write(
            f"LP adaptation steps: "
            f"{LP_ADAPT_STEPS}\n"
        )

        # --------------------------------------------------------
        # Data sizes
        # --------------------------------------------------------

        f.write(
            "\nLP-MIRROR DATA REDUCTION\n"
        )

        f.write("-" * 72 + "\n")

        f.write(
            f"Original training samples: "
            f"{original_train_size}\n"
        )

        f.write(
            f"Selected training samples: "
            f"{selected_train_size}\n"
        )

        f.write(
            f"Retention ratio [%]: "
            f"{100 * retention_ratio:.10f}\n"
        )

        f.write(
            f"Reduction ratio [%]: "
            f"{100 * (1 - retention_ratio):.10f}\n"
        )

        f.write(
            f"Selected input values per epoch "
            f"for final LSTM: "
            f"{selected_input_values}\n"
        )

        # --------------------------------------------------------
        # Timing
        # --------------------------------------------------------

        f.write(
            "\nDURATION\n"
        )

        f.write("-" * 72 + "\n")

        f.write(
            f"LP-Mirror selection time [s]: "
            f"{lp_selection_duration:.10f}\n"
        )

        f.write(
            f"MOTPE HPO time [s]: "
            f"{motpe_duration:.10f}\n"
        )

        f.write(
            f"Total LP-Mirror + MOTPE "
            f"optimization time [s]: "
            f"{total_optimization_duration:.10f}\n"
        )

        f.write(
            f"Total end-to-end duration "
            f"including online evaluation [s]: "
            f"{total_end_to_end_duration:.10f}\n"
        )

        # --------------------------------------------------------
        # Pareto / final config
        # --------------------------------------------------------

        f.write(
            "\nFINAL MOTPE SOLUTION\n"
        )

        f.write("-" * 72 + "\n")

        f.write(
            f"Pareto solutions: "
            f"{len(pareto_trials)}\n"
        )

        f.write(
            f"Selected trial: "
            f"{best_trial.number}\n"
        )

        f.write(
            f"Validation RMSE: "
            f"{best_trial.values[0]:.10f}\n"
        )

        f.write(
            f"Candidate training time [s]: "
            f"{best_trial.values[1]:.10f}\n"
        )

        for key, value in (
            best_architecture.items()
        ):

            f.write(
                f"{key}: {value}\n"
            )

        # --------------------------------------------------------
        # Every raw RMSE
        # --------------------------------------------------------

        f.write(
            "\nRAW-SCALE RMSE FOR EACH "
            "ONLINE TIME SERIES\n"
        )

        f.write("=" * 72 + "\n")

        for row in online_results:

            f.write(
                f"Series "
                f"{row['series_id']}: "
                f"{row['rmse_raw']:.10f}\n"
            )

        # --------------------------------------------------------
        # RMSE summary
        # --------------------------------------------------------

        f.write(
            "\nRAW-SCALE RMSE SUMMARY\n"
        )

        f.write("=" * 72 + "\n")

        f.write(
            f"Number of time series: "
            f"{len(raw_rmse_values)}\n"
        )

        f.write(
            f"Mean RMSE: "
            f"{np.mean(raw_rmse_values):.10f}\n"
        )

        f.write(
            f"Std RMSE: "
            f"{np.std(raw_rmse_values, ddof=1):.10f}\n"
        )

        f.write(
            f"Median RMSE: "
            f"{np.median(raw_rmse_values):.10f}\n"
        )

        f.write(
            f"Min RMSE: "
            f"{np.min(raw_rmse_values):.10f}\n"
        )

        f.write(
            f"Max RMSE: "
            f"{np.max(raw_rmse_values):.10f}\n"
        )

    # ============================================================
    # 8. Console summary
    # ============================================================

    print("\n" + "=" * 80)

    print(
        f"COMPLETED: {experiment_name}"
    )

    print("=" * 80)

    print(
        "LP-Mirror selected:",
        selected_train_size,
        "/",
        original_train_size,
    )

    print(
        "Retention [%]:",
        100 * retention_ratio,
    )

    print(
        "MOTPE trials:",
        len(study.trials),
    )

    print(
        "Total optimization time [s]:",
        total_optimization_duration,
    )

    print(
        "Mean raw online RMSE:",
        np.mean(raw_rmse_values),
    )

    print(
        "Saved:",
        txt_path,
    )

    # ============================================================
    # 9. Cleanup
    # ============================================================

    best_instance.model = (
        best_instance.model.to("cpu")
    )

    trial_models.clear()

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "experiment":
            experiment_name,

        "original_train_size":
            original_train_size,

        "selected_train_size":
            selected_train_size,

        "retention_ratio":
            retention_ratio,

        "lp_selection_duration":
            lp_selection_duration,

        "motpe_duration":
            motpe_duration,

        "total_optimization_duration":
            total_optimization_duration,

        "total_end_to_end_duration":
            total_end_to_end_duration,

        "best_architecture":
            best_architecture,

        "best_validation_rmse":
            float(
                best_trial.values[0]
            ),

        "best_training_time":
            float(
                best_trial.values[1]
            ),

        "raw_rmse_values":
            raw_rmse_values,

        "normalized_rmse_values":
            normalized_rmse_values,

        "txt_path":
            txt_path,
    }

In [ ]:
# ============================================================
# Sanity check: exact subsets used for scalability
# ============================================================

offline_length = int(
    OFFLINE_FRACTION
    * SERIES_LENGTH
)

train_end = int(
    HPO_SPLIT
    * offline_length
)

expected_lp_candidates_per_client = (
    train_end
    - LP_SEQ_LEN
)

print(
    "Offline length per client:",
    offline_length,
)

print(
    "LP-Mirror source segment ends at:",
    train_end,
)

print(
    "LP-Mirror candidate samples per client:",
    expected_lp_candidates_per_client,
)

for n_series in N_VALUES:

    data_n = (
        all_raspberry_cpu[
            :n_series
        ]
    )

    print(
        f"N={n_series:4d} | "
        f"full={data_n.shape} | "
        f"offline=({n_series}, {offline_length}) | "
        f"online=({n_series}, "
        f"{SERIES_LENGTH - offline_length}) | "
        f"expected LP candidates="
        f"{n_series * expected_lp_candidates_per_client}"
    )

In [ ]:
# ============================================================
# RUN CLIENT-COUNT SCALABILITY
#
# Fixed:
#   T = 500
#
# Vary:
#   N = 10, 100, 500, 1000
#
# N = 60 is NOT rerun because it already exists.
# ============================================================

part_a_results = []

for n_series in N_VALUES:

    print(
        "\n" + "#" * 80
    )

    print(
        f"SCALABILITY RUN: "
        f"N={n_series}, "
        f"T={SERIES_LENGTH}"
    )

    print(
        "#" * 80
    )

    # ============================================================
    # Same first-N selection used for MODE scalability
    # ============================================================

    split_data = (
        all_raspberry_cpu[
            :n_series
        ].copy()
    )

    experiment_name = (
        f"RASPBERRY_CPU_N_{n_series}"
    )

    # ============================================================
    # Run complete LP-Mirror + MOTPE pipeline
    # ============================================================

    result = (
        run_lp_mirror_motpe_experiment(
            experiment_name=experiment_name,
            data=split_data,
            output_folder=OUTPUT_FOLDER,
        )
    )

    # ============================================================
    # Sanity check LP-Mirror source sample count
    # ============================================================

    expected_original_size = (
        n_series
        * expected_lp_candidates_per_client
    )

    if (
        result["original_train_size"]
        != expected_original_size
    ):

        raise ValueError(
            f"Unexpected LP-Mirror candidate count "
            f"for N={n_series}: "
            f"got "
            f"{result['original_train_size']}, "
            f"expected "
            f"{expected_original_size}."
        )

    # ============================================================
    # Small remaining pipeline overhead
    # ============================================================

    other_optimization_sec = max(
        0.0,
        result[
            "total_optimization_duration"
        ]
        - result[
            "lp_selection_duration"
        ]
        - result[
            "motpe_duration"
        ],
    )

    # ============================================================
    # Summary row
    # ============================================================

    summary_row = {

        "n_series":
            int(n_series),

        "series_length":
            int(SERIES_LENGTH),

        "offline_length":
            int(offline_length),

        "online_length":
            int(
                SERIES_LENGTH
                - offline_length
            ),

        "total_full_values":
            int(
                n_series
                * SERIES_LENGTH
            ),

        "total_offline_values":
            int(
                n_series
                * offline_length
            ),

        "original_train_size":
            int(
                result[
                    "original_train_size"
                ]
            ),

        "selected_train_size":
            int(
                result[
                    "selected_train_size"
                ]
            ),

        "retention_percent":
            float(
                100.0
                * result[
                    "retention_ratio"
                ]
            ),

        "lp_selection_sec":
            float(
                result[
                    "lp_selection_duration"
                ]
            ),

        "motpe_hpo_sec":
            float(
                result[
                    "motpe_duration"
                ]
            ),

        "other_optimization_sec":
            float(
                other_optimization_sec
            ),

        "total_optimization_sec":
            float(
                result[
                    "total_optimization_duration"
                ]
            ),

        "total_end_to_end_sec":
            float(
                result[
                    "total_end_to_end_duration"
                ]
            ),

        "best_validation_rmse":
            float(
                result[
                    "best_validation_rmse"
                ]
            ),

        "best_training_time_sec":
            float(
                result[
                    "best_training_time"
                ]
            ),

        "mean_raw_rmse":
            float(
                np.mean(
                    result[
                        "raw_rmse_values"
                    ]
                )
            ),

        "std_raw_rmse":
            float(
                np.std(
                    result[
                        "raw_rmse_values"
                    ],
                    ddof=1,
                )
            ),

        "median_raw_rmse":
            float(
                np.median(
                    result[
                        "raw_rmse_values"
                    ]
                )
            ),

        "min_raw_rmse":
            float(
                np.min(
                    result[
                        "raw_rmse_values"
                    ]
                )
            ),

        "max_raw_rmse":
            float(
                np.max(
                    result[
                        "raw_rmse_values"
                    ]
                )
            ),

        "txt_path":
            result[
                "txt_path"
            ],
    }

    part_a_results.append(
        summary_row
    )

    # ============================================================
    # Save after EACH successful N
    # ============================================================

    part_a_summary_df = (
        pd.DataFrame(
            part_a_results
        )
        .sort_values(
            "n_series"
        )
        .reset_index(
            drop=True
        )
    )

    summary_path = os.path.join(
        OUTPUT_FOLDER,
        "Part_A_summary.csv",
    )

    part_a_summary_df.to_csv(
        summary_path,
        index=False,
    )

    print(
        "\nSaved current summary:",
        summary_path,
    )

    display(
        part_a_summary_df
    )

    # ============================================================
    # Cleanup before next N
    # ============================================================

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

summary_path = os.path.join(
    OUTPUT_FOLDER,
    "Part_A_summary.csv",
)

part_a_summary_df = pd.read_csv(
    summary_path
)

part_a_summary_df = (
    part_a_summary_df
    .sort_values(
        "n_series"
    )
    .reset_index(
        drop=True
    )
)

display(
    part_a_summary_df
)